# Tropical cyclone track map and time series
## ATM 433/533 Fall 2026 Week 2

## Overview
In this notebook, you will create visualizations of tropical cyclone data, using Pandas, Matplotlib and Cartopy.

### Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.dates import DateFormatter, AutoDateLocator, HourLocator, DayLocator, MonthLocator

### Part 1: Minimum central sea-level pressure and maximum wind speed of Hurricane Helene (2024)
Open the `csv` file containing 6-hourly data from Helene, as archived in [NHC's HURDAT](https://www.nhc.noaa.gov/data/).

Use Matplotlib and create a single Figure with two subplots (i.e., *Axes*), one on top of the other:
1. On the top subplot, plot date and time on the x-axis, and central sea-level pressure in hPa on the y-axis.
2. On the bottom subplot, plot maximum sustained wind speed in mph on the y-axis.
3. Save your figure as a PNG

Set the name and year of the tropical cyclone to use in figure captions

In [ ]:
tc_name = 'Helene'
tc_year = '2024'

Read in the CSV file as a Pandas `DataFrame`

In [ ]:
df = pd.read_csv('/spare11/atm533/data/helene_2024.csv')

Examine the `DataFrame`

In [ ]:
df

Read in columns, termed as `Series` in Pandas, of parameters of interest:
1. Latitude (deg)
1. Longitude (deg)
1. Central SLP (hPa)
1. Maximum wind speed (kts)
1. Date and time (format: YYYY-MM-DD HH:MM:SS; time zone: UTC)

In [ ]:
lat = df.Lat
lon = df.Lon
slp = df.Min_SLP
wspd = df.Max_Speed
dattim = df.Time

Create a `Figure` with two subplots (aka, `Axes`) and make two line graphs. On the top (bottom) `Axes`, plot SLP (max wind) versus date/time.

In [ ]:
fig = plt.figure(figsize=(12,9))

ax1 = fig.add_subplot (2,1,1)
ax1.plot(dattim, slp)

ax2 = fig.add_subplot(2,1,2)
ax2.plot(dattim, wspd);

To make this figure, we have extracted the series from the Pandas Dataframe. Instead we could use the Pandas plotting functionality which cleans up the dates on the x-axis and adds a legend.

In [ ]:
fig = plt.figure(figsize=(12, 9))

ax1 = fig.add_subplot(2, 1, 1)
df.plot(x='Time', y='Min_SLP', ax=ax1)

ax2 = fig.add_subplot(2, 1, 2)
df.plot(x='Time', y='Max_Speed', ax=ax2);

A quick and easy way to make the plot better looking is to import and apply the [Seaborn](https://seaborn.pydata.org/) package.

In [ ]:
import seaborn as sns

# Activate the default seaborn theme globally
sns.set_theme() 

In [ ]:
fig = plt.figure(figsize=(12, 9))

ax1 = fig.add_subplot(2, 1, 1)
df.plot(x='Time', y='Min_SLP', ax=ax1)

ax2 = fig.add_subplot(2, 1, 2)
df.plot(x='Time', y='Max_Speed', ax=ax2);

Add a title and axis labels

In [ ]:
fig = plt.figure(figsize=(12, 9))
fig.suptitle(f'{tc_name} ({tc_year}) Min. SLP, Max. Wind', fontsize=16)

ax1 = fig.add_subplot(2, 1, 1)
df.plot(x='Time', y='Min_SLP', ax=ax1)
ax1.set_xlabel('Date and time')
ax1.set_ylabel('SLP (hPa)')

ax2 = fig.add_subplot(2, 1, 2)
df.plot(x='Time', y='Max_Speed', ax=ax2)
ax2.set_xlabel('Date and time')
ax2.set_ylabel('Windspeed (kts)');

<div class="admonition alert alert-info">
    <p class="admonition-title" style="font-weight:bold">A better visualization:</p>
   Rather than placing SLP and wind speed in their own subplots, let's have them share a single subplot. We will need to add a legend, and also have a separate y-axis for each variable.
</div>


In [ ]:
fig = plt.figure(figsize=(12, 9))
fig.suptitle(f'{tc_name} ({tc_year}) Min. SLP, Max. Wind', fontsize=16)

ax1_color='blue'
ax2_color='red'

ax1 = fig.add_subplot(1, 1, 1)
df.plot(x='Time', y='Min_SLP', ax=ax1, color=ax1_color, label='SLP', legend=False)
ax1.set_xlabel('Date and time')
ax1.set_ylabel('SLP (hPa)', color=ax1_color)

ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis
df.plot(x='Time', y='Max_Speed', ax=ax2, color=ax2_color, label='Windspeed', legend=False)
#ax2.set_xlabel('Date and time') # we do not need an additional x-axis specified, it is handled in the x-label with ax1
ax2.set_ylabel('Windspeed (kts)', color=ax2_color)

# Because ax1 and ax2 are separate axes, each creates its own legend.
# We set the legend to False for the two axes and generate our own by asking matplotlib for the plotted objects and labels
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2);

There are still a few things we can improve:
1. Re-cast the x-axis from strings to `Datetime` objects, using a handy **Pandas** method
1. Re-format the x-axis labels, now that they are `Datetime` objects
1. Distinguish the y-axis gridlines

In [ ]:
dattim_dt = pd.to_datetime(dattim,format="%Y-%m-%d %H:%M:%S")

# Alternatively, we can create a new column using the known format of the 'Time' column
df['Time_dt'] = pd.to_datetime(
    df['Time'],
    format='%Y-%m-%d %H:%M:%S'
)

In [ ]:
fig = plt.figure(figsize=(12, 9))
fig.suptitle(f'{tc_name} ({tc_year}) Min. SLP, Max. Wind', fontsize=16)

ax1_color='blue'
ax2_color='red'

# Plot ax1
ax1 = fig.add_subplot(1, 1, 1)
df.plot(x='Time_dt', y='Min_SLP', ax=ax1, color=ax1_color, label='SLP', legend=False)
ax1.set_xlabel('Date and time')
ax1.set_ylabel('SLP (hPa)', color=ax1_color)
ax1.set_ylim(900,1020)

# Format ax1
ax1.tick_params(axis='y', labelcolor=ax1_color)
ax1.tick_params(axis='x', labelrotation=0) # optional: set label rotation explicitly
ax1.grid(color=ax1_color, axis='y', linestyle='dashed', linewidth=0.5)
ax1.xaxis.set_major_locator(DayLocator(interval=1))
dateFmt = DateFormatter('%b %d')
ax1.xaxis.set_major_formatter(dateFmt)

# Plot ax2 
ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis
df.plot(x='Time_dt', y='Max_Speed', ax=ax2, color=ax2_color, label='Windspeed', legend=False)
ax2.set_ylabel('Windspeed (kts)', color=ax2_color)
ax2.set_ylim(0, 160)

# Format ax2
ax2.tick_params(axis='y', labelcolor=ax2_color)
ax2.grid(color=ax2_color, axis='y', linestyle='dotted', linewidth=0.3)


# Because ax1 and ax2 are separate axes, each creates its own legend.
# We set the legend to False for the two axes and generate our own by asking matplotlib for the plotted objects and labels
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2);

# Could also set the title to the axis rather than the figure to remove some empty space (comment out fig.suptitle above)
#ax1.set_title(f'{tc_name} ({tc_year}) Min. SLP, Max. Wind', fontsize=16);

Save the figure as a PNG

In [ ]:
fig.savefig(f'{tc_name}{tc_year}_TimeSeries_SLP_WSPD.png')

### Part 2: Plot the center of Helene on a map
1. Use Matplotlib and Cartopy to plot the locations of Helene for the same time period used in Part 1.
2. Save your figure as a PNG

Define an object pointing to the Cartopy coordinate reference system in which the dataset is based. Since the data is in lat-lon coordinates, we'll use `PlateCarree`.

In [ ]:
projData = ccrs.PlateCarree()

Define the bounds over which to plot the data

In [ ]:
mapBounds = [-90,-50,10,40]

Below we will use the Pandas Series lat/lon rather than the native Pandas Matplotlib plotting (i.e., df.plot) to show multiple methods

In [ ]:
fig = plt.figure(figsize=(12,9))
ax = fig.add_subplot (projection=projData)

ax.set_extent(mapBounds, crs=projData)
ax.set_title(f'Center locations of {tc_name} ({tc_year})')

gl = ax.gridlines(draw_labels=True, linewidth=2, color='gray', alpha=0.5, linestyle='--')

ax.set_facecolor(cfeature.COLORS['water'])
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle='--')
ax.add_feature(cfeature.LAKES, alpha=0.5)
ax.add_feature(cfeature.STATES)
ax.add_feature(cfeature.RIVERS)


# lon, lat are Pandas series extracted near the top of this notebook
ax.plot(lon,lat); # alternative to df.plot(x='Lon', y='Lat', ax=ax)

Let's improve the look of the figure by making the track line easier to see

In [ ]:
fig = plt.figure(figsize=(12,9))
ax = fig.add_subplot (projection=projData)

ax.set_extent(mapBounds, crs=projData)
ax.set_title(f'Center locations of {tc_name} ({tc_year})')

gl = ax.gridlines(draw_labels=True, linewidth=2, color='gray', alpha=0.5, linestyle='--')

ax.set_facecolor(cfeature.COLORS['water'])
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle='--')
ax.add_feature(cfeature.LAKES, alpha=0.5)
ax.add_feature(cfeature.STATES)
ax.add_feature(cfeature.RIVERS)

ax.plot(lon,lat, color='darkgreen', marker='o', linestyle='dashed', linewidth=2, markersize=12);

We'll further improve the figure by using tropical cyclone symbols to mark the locations; the [tcmarkers](https://github.com/abrammer/tc_markers) package works nicely!

In [ ]:
import tcmarkers

In [ ]:
fig = plt.figure(figsize=(12,9))
ax = fig.add_subplot (projection=projData)

ax.set_extent(mapBounds, crs=projData)
ax.set_title(f'Center locations of {tc_name} ({tc_year})')

gl = ax.gridlines(draw_labels=True, linewidth=2, color='gray', alpha=0.5, linestyle='--')

ax.set_facecolor(cfeature.COLORS['water'])
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle='--')
ax.add_feature(cfeature.LAKES, alpha=0.5)
ax.add_feature(cfeature.STATES)
ax.add_feature(cfeature.RIVERS)

ax.plot(lon,lat, color='darkgreen')

marker_kwargs = {'s': 40, 'color':'darkgreen', 'edgecolor':'darkgreen'}

# make the symbol wind speed dependent
for idx, value in enumerate (wspd):
    if (value < 34): # tropical depression (TD)
        sym = tcmarkers.TD
        if (lat[idx] < 0):
            sym = tcmarkers.SH_TD
    elif (value < 64): # tropical storm (TS)
        sym = tcmarkers.TS
        if (lat[idx] < 0):
            sym = tcmarkers.SH_TS
    else: # Hurricane (HU)
        sym = tcmarkers.HU
        if (lat[idx] < 0):
            sym = tcmarkers.SH_HU
    
    ax.scatter(lon[idx], lat[idx], marker=sym, **marker_kwargs);


Save the figure as a PNG

In [ ]:
fig.savefig(f'{tc_name}_{tc_year}_TrackMap.png')

<div class="admonition alert alert-danger">
    <p class="admonition-title" style="font-weight:bold">REMINDER</p>
    Remember to save, close and shutdown your notebook when you are not actively developing it!
</div>


## References
1. [HURDAT: Landsea and Franklin, 2013](https://doi.org/10.1175/MWR-D-12-00254.1)
1. [Matplotlib Introduction, ATM350](https://www.atmos.albany.edu/facstaff/ktyle/atm350/core/week6/01_MatplotlibIntro.html)
1. [Pandas Introduction, ATM350](https://www.atmos.albany.edu/facstaff/ktyle/atm350/core/week7/01_Pandas.html)
1. [Pandas II, ATM350](https://www.atmos.albany.edu/facstaff/ktyle/atm350/core/week7/02_Pandas.html)
1. [Cartopy Introduction, ATM350](https://www.atmos.albany.edu/facstaff/ktyle/atm350/core/week8/01_Cartopy_Introduction.html)
1. [Cartopy II, ATM350](https://www.atmos.albany.edu/facstaff/ktyle/atm350/core/week9/02_Cartopy_NYSMesonet.html)
1. [Dates and Times, ATM350](https://www.atmos.albany.edu/facstaff/ktyle/atm350/core/week8/Datetime.html)
